In [1]:
# pip install langchain langchain-community langchain-chroma langchain-huggingface chromadb sentence-transformers

In [2]:
import json

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.chains import RetrievalQA

import warnings
warnings.filterwarnings("ignore")

C:\Users\FGstore\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1) Load JSON movie catalog
with open("movies.json", "r", encoding="utf-8") as f:
    catalog = json.load(f)

In [4]:
# 2) Convert JSON to LangChain Documents
docs = []

# Create documents for individual movies
for genre, movies in catalog.items():
    for movie in movies:
        title       = movie.get("title", "")
        year        = movie.get("year", "")
        director    = movie.get("director", "")
        cast        = ", ".join(movie.get("cast", []))
        description = movie.get("description", "")
        rating      = movie.get("rating", "")
        duration    = movie.get("duration_min", "")
        language    = movie.get("language", "")

        # Rich text that the embedding model will encode
        content = f"""
Genre: {genre}
Title: {title} ({year})
Director: {director}
Cast: {cast}
Description: {description}
IMDb Rating: {rating}/10
Duration: {duration} minutes
Language: {language}
"""
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "genre":    genre,
                    "title":    title,
                    "year":     year,
                    "director": director,
                    "rating":   rating,
                    "language": language,
                    "type":     "movie"
                }
            )
        )


print(f"Created {len(docs)} documents ({len(docs)//2} movies + {len(docs)//2} genre docs).")
print("\nSample document:")
print(docs[0].page_content)
print("Metadata:", docs[0].metadata)

Created 31 documents (15 movies + 15 genre docs).

Sample document:

Genre: Action
Title: Mad Max: Fury Road (2015)
Director: George Miller
Cast: Tom Hardy, Charlize Theron
Description: In a post-apocalyptic wasteland, Max teams up with a rebel warrior to escape a tyrannical warlord.
IMDb Rating: 8.1/10
Duration: 120 minutes
Language: English

Metadata: {'genre': 'Action', 'title': 'Mad Max: Fury Road', 'year': 2015, 'director': 'George Miller', 'rating': 8.1, 'language': 'English', 'type': 'movie'}


In [5]:
# 3) Embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [6]:
# 4) Build and persist Chroma vector store (delete and recreate to include genre documents)
import os
import shutil

if os.path.exists("./chroma_movies_db"):
    shutil.rmtree("./chroma_movies_db")  # Delete old database

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="movies_collection",
    persist_directory="./chroma_movies_db"
)

In [7]:
# 5) Retriever — optimized search with more context
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}  # Get more documents to ensure genre docs are included
)

# Quick sanity check
test_results = retriever.invoke("psychological thriller with a twist ending")
test_results

[Document(id='4d348313-fd71-4f8e-b06e-fedee9f3de0f', metadata={'title': 'The Shining', 'language': 'English', 'year': 1980, 'rating': 8.4, 'genre': 'Horror', 'type': 'movie', 'director': 'Stanley Kubrick'}, page_content='\nGenre: Horror\nTitle: The Shining (1980)\nDirector: Stanley Kubrick\nCast: Jack Nicholson, Shelley Duvall\nDescription: A writer becomes the winter caretaker of an isolated hotel and descends into madness.\nIMDb Rating: 8.4/10\nDuration: 146 minutes\nLanguage: English\n'),
 Document(id='324bca72-85b3-4821-beac-88470d06d573', metadata={'type': 'movie', 'rating': 7.7, 'director': 'Jordan Peele', 'year': 2017, 'genre': 'Horror', 'title': 'Get Out', 'language': 'English'}, page_content="\nGenre: Horror\nTitle: Get Out (2017)\nDirector: Jordan Peele\nCast: Daniel Kaluuya, Allison Williams\nDescription: A Black man meets his white girlfriend's parents for a weekend, uncovering a disturbing secret.\nIMDb Rating: 7.7/10\nDuration: 104 minutes\nLanguage: English\n"),
 Documen

In [8]:
# 6) Hugging Face LLM
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 250,
        "temperature": 0.3,
        "return_full_text": False
    }
)


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Device set to use cuda:0


In [9]:
# 7) Prompt template — movie assistant persona
prompt = PromptTemplate(
    template="""
You are a knowledgeable movie expert assistant.

Use ONLY the movie catalog context below to answer the question.

If the user asks about movies in a genre (e.g., "what sci-fi movies do you have"), 
provide a comprehensive list of all available movies in that genre with their ratings and descriptions.

If the user asks about a specific movie, provide detailed information about that movie.

When recommending movies, mention the genre, year, and IMDb rating if available.

If the answer cannot be found in the catalog, say:
"I don't have that information in the catalog."

Movie catalog context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

In [10]:
# 8) RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

In [11]:
# Start an interactive chat loop for menu queries
while True:
    question = input("\nAsk about the menu: ")

    print(f"\nquestion: {question}")

    if question.lower() in ["exit", "quit", "q"]:
        break

    result = qa_chain.invoke({"query": question})

    print("\nDocuments Retrieved:")
    if result["source_documents"]:
        for i, doc in enumerate(result["source_documents"], 1):
            print(f"  {i}. {doc.metadata}")
    else:
        print("No documents found!")

    print("\nBot:", result["result"])


question: need great movie

Documents Retrieved:
  1. {'rating': 8.1, 'genre': 'Thriller', 'director': 'Denis Villeneuve', 'type': 'movie', 'year': 2013, 'language': 'English', 'title': 'Prisoners'}
  2. {'rating': 8.1, 'year': 2015, 'type': 'movie', 'director': 'George Miller', 'genre': 'Action', 'language': 'English', 'title': 'Mad Max: Fury Road'}
  3. {'genre': 'Action', 'year': 2014, 'rating': 7.4, 'title': 'John Wick', 'director': 'Chad Stahelski', 'language': 'English', 'type': 'movie'}
  4. {'type': 'movie', 'title': '13th', 'genre': 'Documentary', 'director': 'Ava DuVernay', 'year': 2016, 'language': 'English', 'rating': 8.2}
  5. {'genre': 'Horror', 'director': 'Jordan Peele', 'rating': 7.7, 'language': 'English', 'type': 'movie', 'title': 'Get Out', 'year': 2017}
  6. {'language': 'English', 'genre': 'Thriller', 'rating': 8.1, 'year': 2014, 'type': 'movie', 'title': 'Gone Girl', 'director': 'David Fincher'}
  7. {'language': 'English', 'rating': 8.1, 'title': 'The Grand Bud